# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os   
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [6]:
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
MODEL = "llama3.2"

In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-productio

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer

In [10]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 6 relevant links


{'links': [{'type': 'About page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Company/Careers page', 'url': 'https://edwarddonner.com/'},
  {'type': 'Blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn Profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter Profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook Profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 7 relevant links


{'links': [{'type': 'Company page', 'url': 'https://huggingface.co'},
  {'type': 'About us page', 'url': 'https://huggingface.co/brand'},
  {'type': 'Careers/Jobs page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'Documentation link', 'url': 'https://docs.huggingface.co'},
  {'type': 'Blog link', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub repository', 'url': 'https://github.com/huggingface'},
  {'type': 'Discord channel', 'url': 'https://join.discord.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [21]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        try:
            result += fetch_website_contents(link["url"])
        except Exception as e:
            result += f"(Could not fetch: {e})"
    return result

In [22]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 4 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
openbmb/MiniCPM5-1B
Updated
3 days ago
•
23.6k
•
532
meituan-longcat/LongCat-Video-Avatar-1.5
Updated
3 days ago
•
385
bytedance-research/Lance
Updated
1 day ago
•
2.74k
•
968
nvidia/LocateAnything-3B
Updated
2 days ago
•
7.86k
•
326
HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated

In [34]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [35]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [36]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nopenbmb/MiniCPM5-1B\nUpdated\n3 days ago\n•\n23.6k\n•\n533\nmeituan-longcat/LongCat-Video-Avatar-1.5\nUpdated\n3 days ago\n•\n

In [37]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [38]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 6 relevant links


**Welcome to Hugging Face: Where AI Collaboration Meets Unlimited Possibilities**

Imagine a world where machine learning models are created, shared, and improved upon in a matter of seconds. A world where the brightest minds in AI come together to tackle complex problems and push the boundaries of innovation. This is the world that Hugging Face is building.

**About Us**

Hugging Face is the go-to platform for machine learning professionals around the globe. We're passionate about collaboration, innovation, and empowering developers to build amazing applications using AI. Our mission is to make machine learning more accessible, efficient, and fun for everyone.

**What We Offer**

* **2 Million+ Models**: Browse our vast library of pre-trained models, updated every second, to find the perfect solution for your project.
* **Collaboration Platform**: Host and collaborate on unlimited public projects with a community of like-minded individuals.
* **Data Resources**: Access 500,000+ datasets, curated and organized for easy discovery and use in your applications.
* ** AI-Driven Tools**: Explore our range of cutting-edge tools, including audio-driven talking-head video generation, image editing, and more.

**Our Community**

Join the 100,000+ strong Hugging Face community, where you can share ideas, get feedback, and learn from experts in the field. Say hello to like-minded individuals who are passionate about harnessing the power of AI for greater good.

**Careers & Opportunities**

Ready to be part of something groundbreaking? Our team is growing rapidly, with opportunities across software development, research, and more. Check out our [careers page](link) to explore the many ways you can contribute to Hugging Face's mission.

**Why Join Us?**

* **Innovative Atmosphere**: Work alongside brilliant minds who are pushing the boundaries of AI innovation.
* **Autonomy & Freedom**: Contribute to projects that truly matter, without bureaucratic hurdles or unnecessary bureaucracy.
* **Collaborative Culture**: Build lasting relationships with fellow developers and thought leaders who share your passion for machine learning.

"Building a better future, one model at a time."

Join the Hugging Face revolution today!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [31]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [32]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 6 relevant links


**Welcome to Hugging Face: Where AI Meets Community**

Hey there! Are you ready to unlock the future of artificial intelligence? Look no further than Hugging Face, the dynamic platform where machine learning enthusiasts come together to build, discover, and explore the latest advancements in AI.

**Our Mission**

At Hugging Face, we believe that AI should be accessible to everyone, regardless of their technical expertise. We're dedicated to creating an inclusive community that fosters collaboration, innovation, and creativity. Whether you're a researcher, student, or simply interested in staying ahead of the curve, we've got you covered.

**What Sets Us Apart**

Our platform boasts an incredible 2 million+ pre-trained models across various applications, from natural language processing to computer vision and more. With Hugging Face, you can:

Explore state-of-the-art models and algorithms
Discover innovative applications in your browser or on-premises
Host and collaborate with unlimited public datasets

**Community Features**

Our community is the heart of Hugging Face. Join us to:

Network with experts and like-minded individuals worldwide 
Share knowledge and resources through our forum, Reddit, and GitHub repos
Stay up-to-date with the latest blogs, papers, and research insights from top AI researchers

**Join Our Journey**

Ready to be part of the AI revolution? Browse our 2 million+ models, explore innovative applications, and join our community today!

**Careers & Careers Resources**

Looking for new career opportunities in AI or want to learn more about a specific field? Check out our careers resources:

* Explore job descriptions and company culture on our Careers page
* Learn from industry experts and top researchers through webinars and workshops

Join us at Hugging Face and be part of the AI community building the future!

In [33]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 8 relevant links


# Welcome to Hugging Face: Where AI Collaboration Meets Creativity

At Hugging Face, we believe that the future of Artificial Intelligence (AI) lies at the intersection of collaboration and creativity. That's why we've built a platform that empowers machine learning practitioners to work together, share knowledge, and push the boundaries of what's possible.

## Our Mission

Our mission is to create a space where AI researchers and developers can come together to build amazing things. We achieve this by providing an open-source platform for building, testing, and deploying machine learning models. With Hugging Face, you can explore 2 million+ pre-trained models, collaborate on projects with like-minded individuals, and access to cutting-edge AI tools and technologies.

## Our Community

Our community is at the heart of everything we do. Comprising of over 1 million machine learning practitioners, researchers, and developers from around the world, our community is the driving force behind innovation in the field of AI. With Hugging Face, you can connect with like-minded individuals, share knowledge, and stay updated on the latest developments in the rapidly evolving landscape of AI.

## Our Tools

Hugging Face is home to a wide range of tools and technologies designed to make machine learning more accessible and user-friendly. These include:

* **Transformers**: A popular library for natural language processing (NLP) tasks that has revolutionized the field of NLP.
* **Datasets**: A vast repository of ready-to-use datasets that can be used for training, testing, and validation.
* **Spaces**: A collaboration platform where users can host, launch, and manage their own machine learning models.
* **buckets**: A secure storage system that allows users to store, distribute, and retrieve pre-trained models.

## Our Blog

Stay up-to-date with the latest developments in AI by following our blog. Our experts share insights, trends, and best practices on topics such as NLP, computer vision, and natural language processing.

# Join the Hugging Face Community Today

Whether you're a seasoned researcher or just starting out in machine learning, we invite you to join our community. With Hugging Face, you can:

* Explore 2 million+ pre-trained models
* Collaborate on projects with like-minded individuals
* Stay updated on the latest developments in AI
* Discover cutting-edge AI tools and technologies

Sign up today and become part of a vibrant community that's shaping the future of AI.

# Careers at Hugging Face

Join our team of innovative minds who are passionate about creating amazing things. We're always hiring talented individuals to help drive our mission forward. Check out our job listings and apply today!

# Get in Touch with Us

[Your Company Email Address](mailto:yourcompany email address)
[Your Company Phone Number]
Visit our website at [Your Website URL]

Stay connected with us on social media:

Twitter:[@Your Twitter Handle](https://twitter.com/yourTwitterHandle)
Facebook:[@Your Facebook Page](https://www.facebook.com/@YourFacebookPage)

# Let's Build the Future of AI Together

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>